# Getting Started with SageMaker Notebooks

Welcome to SageMaker Notebooks in Amazon SageMaker Unified Studio! This notebook will guide you through the key features that make data exploration, analysis, and processing seamless across multiple languages, engines, and data sources.

Notebooks provide a unified interface where you can work with local data using Python and SQL, query data from AWS Glue Data Catalog using Athena, process large-scale data with Spark, and even connect to external sources like Snowflake - all in one place. Whether you're doing exploratory data analysis, building data pipelines, or preparing data for machine learning, this notebook environment has you covered.

**Note:** This notebook includes fallback mechanisms for when data sources are unavailable. If you see fallback warnings, follow the instructions provided to resolve the underlying issues.


## Setup: Define Fallback Data and Helper Functions

This cell defines fallback datasets and helper functions used throughout the notebook to ensure smooth execution even when external data sources are unavailable.


In [0]:
import pandas as pd
import numpy as np
import warnings
from typing import Optional, Tuple, Any

# =============================================================================
# FALLBACK DATA DEFINITIONS
# =============================================================================

def get_fallback_churn_data() -> pd.DataFrame:
    """
    Returns a hardcoded sample of churn data that mirrors the structure
    of the S3 churn dataset.
    """
    np.random.seed(42)
    n_samples = 100
    
    states = ['KS', 'OH', 'NJ', 'OK', 'AL', 'MA', 'MO', 'LA', 'WV', 'IN',
              'RI', 'IA', 'MT', 'NY', 'ID', 'VT', 'VA', 'TX', 'FL', 'CO']
    
    data = {
        'state': np.random.choice(states, n_samples),
        'account_length': np.random.randint(1, 250, n_samples),
        'area_code': np.random.choice([408, 415, 510], n_samples),
        'phone': [f'{np.random.randint(100,999)}-{np.random.randint(1000,9999)}' for _ in range(n_samples)],
        'intl_plan': np.random.choice(['yes', 'no'], n_samples, p=[0.1, 0.9]),
        'vmail_plan': np.random.choice(['yes', 'no'], n_samples, p=[0.3, 0.7]),
        'vmail_message': np.random.randint(0, 50, n_samples),
        'day_mins': np.round(np.random.uniform(0, 350, n_samples), 1),
        'day_calls': np.random.randint(0, 165, n_samples),
        'day_charge': np.round(np.random.uniform(0, 60, n_samples), 2),
        'eve_mins': np.round(np.random.uniform(0, 360, n_samples), 1),
        'eve_calls': np.random.randint(0, 170, n_samples),
        'eve_charge': np.round(np.random.uniform(0, 30, n_samples), 2),
        'night_mins': np.round(np.random.uniform(0, 400, n_samples), 1),
        'night_calls': np.random.randint(0, 175, n_samples),
        'night_charge': np.round(np.random.uniform(0, 18, n_samples), 2),
        'intl_mins': np.round(np.random.uniform(0, 20, n_samples), 1),
        'intl_calls': np.random.randint(0, 20, n_samples),
        'intl_charge': np.round(np.random.uniform(0, 5.5, n_samples), 2),
        'custserv_calls': np.random.randint(0, 10, n_samples),
        'churn': np.random.choice(['True.', 'False.'], n_samples, p=[0.15, 0.85])
    }
    
    return pd.DataFrame(data)


def get_fallback_enriched_data() -> pd.DataFrame:
    """
    Returns a hardcoded sample of enriched churn data that mirrors the structure
    of the enriched_table.
    """
    df = get_fallback_churn_data()
    df['total_minutes'] = df['day_mins'] + df['eve_mins'] + df['night_mins'] + df['intl_mins']
    df['total_charges'] = df['day_charge'] + df['eve_charge'] + df['night_charge'] + df['intl_charge']
    return df


# =============================================================================
# HELPER FUNCTIONS FOR ERROR HANDLING
# =============================================================================

def print_fallback_warning(data_source: str, error: Exception, resolution_steps: list):
    """
    Prints a formatted warning message when falling back to sample data.
    """
    print("\n" + "="*80)
    print("⚠️  FALLBACK MODE ACTIVATED")
    print("="*80)
    print(f"\n❌ Failed to load data from: {data_source}")
    print(f"\n📋 Error Details: {type(error).__name__}: {str(error)[:200]}")
    print("\n✅ Using fallback sample data to continue execution.")
    print("\n🔧 To resolve this issue and use real data, try the following:")
    for i, step in enumerate(resolution_steps, 1):
        print(f"   {i}. {step}")
    print("\n📝 After addressing the issue, re-run this cell to load real data.")
    print("="*80 + "\n")


def print_success_message(data_source: str, row_count: int):
    """
    Prints a success message when data is loaded successfully.
    """
    print(f"✅ Successfully loaded {row_count} rows from {data_source}")


# Track which data sources are using fallbacks
FALLBACK_STATUS = {
    'churn_s3': False,
    'enriched_table': False,
    's3_tables': False,
    'spark_session': False
}

print("✅ Fallback data and helper functions initialized successfully.")

## 1. Polyglot Programming - Work with Python and SQL on Local Data

One of the most powerful features of notebooks in SageMaker is the ability to seamlessly switch between Python and SQL within the same notebook. You can leverage the strengths of each language for different tasks - use pandas for data manipulation and PySpark for complex transformations, DuckDB SQL for local exploration or Athena for larger queries, and built-in visualization tools for insights.

Let's start by creating some sample data and exploring it using different approaches.


### 1.1 Working with pandas

pandas is the go-to library for data analysis in Python. Let's create a sample sales dataset and perform some basic exploration.


In [0]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Create sample sales data
np.random.seed(42)
dates = [datetime(2024, 1, 1) + timedelta(days=x) for x in range(100)]
products = ['Laptop', 'Phone', 'Tablet', 'Monitor', 'Keyboard']
regions = ['North', 'South', 'East', 'West']

sales_data = {
    'date': np.random.choice(dates, 200),
    'product': np.random.choice(products, 200),
    'region': np.random.choice(regions, 200),
    'quantity': np.random.randint(1, 10, 200),
    'unit_price': np.random.uniform(50, 1500, 200).round(2)
}

sales_df = pd.DataFrame(sales_data)
sales_df['total_sales'] = (sales_df['quantity'] * sales_df['unit_price']).round(2)

# Display first few rows
sales_df.head(3)

In [0]:
# Basic pandas operations
print("Dataset Shape:", sales_df.shape)
print("\nSummary Statistics:")
sales_df[['quantity', 'unit_price', 'total_sales']].describe()

### 1.2 Local DataFrame SQL with DuckDB

Now let's use an SQL Cell to query the same pandas DataFrame using via DuckDB. The SQL Cell can directly query pandas DataFrames in memory, giving you the flexibility to use SQL syntax when it's more convenient. To query local dataframes create an SQL cell and select `Dataframes` as the data source. You can then `SELECT * FROM <your dataframe var>`.


In [0]:
-- Query the pandas DataFrame using SQL
-- The Dataframe can be directly referenced by name
SELECT 
    product,
    region,
    COUNT(*) as "num_transactions",
    SUM(quantity) as "total_quantity",
    ROUND(SUM(total_sales), 2) as "total_revenue"
FROM sales_df
GROUP BY product, region
ORDER BY total_revenue DESC
LIMIT 3

### 1.3 Visualizing Data

Let's create some visualizations to better understand the sales patterns across products and regions.

Visualization is built into each cell. When looking at the table output for a cell select the Chart button to the top-right between the code and results. Feel free to select and play with it in the above cell.


In addition to our built-in visualization, SageMaker Notebooks also have support for other widely-used visualization libraries like matplotlib.


In [0]:
import matplotlib.pyplot as plt

# Sales by product
product_sales = sales_df.groupby('product')['total_sales'].sum().sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart for product sales
product_sales.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Total Sales by Product', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Total Sales ($)')
axes[0].set_ylabel('Product')

# Pie chart for regional distribution
region_sales = sales_df.groupby('region')['total_sales'].sum()
axes[1].pie(region_sales, labels=region_sales.index, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Sales Distribution by Region', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 2. Work Across Engines

Notebooks can work in multiple engines including local python, local SQL via DuckDB, Athena SQL and Athena PySpark and SparkSQL.

### Queries with Athena (SQL)

Notebooks can seamlessly query data stored in the AWS Glue Data Catalog using Amazon Athena. This allows you to work with data stored in S3 without having to download it locally first.

Let's query the customer_churn sample table from the Glue Data Catalog and explore it.


### 2.1 Reading Data from S3 (with Fallback)

We'll read the churn data directly from S3. If the S3 file is unavailable or access is denied, we'll use fallback sample data.


In [0]:
import boto3
import pandas as pd

# Initialize churn_df as None
churn_df = None
FALLBACK_STATUS['churn_s3'] = False

try:
    region = boto3.Session().region_name
    
    # Read directly from the S3 location using pandas
    s3_data_path = f"s3://sagemaker-example-files-prod-{region}/datasets/tabular/synthetic/churn.txt"
    
    print(f"Attempting to read data from: {s3_data_path}")
    
    # Read the CSV file with pandas
    churn_df = pd.read_csv(s3_data_path)
    
    # Normalize column names: lowercase and replace spaces with underscores
    churn_df.columns = churn_df.columns.str.lower().str.replace(' ', '_').str.replace("'", '').str.replace('?', '')
    
    print_success_message(s3_data_path, len(churn_df))

except FileNotFoundError as e:
    FALLBACK_STATUS['churn_s3'] = True
    churn_df = get_fallback_churn_data()
    print_fallback_warning(
        data_source=f"S3: {s3_data_path}",
        error=e,
        resolution_steps=[
            "Verify the S3 bucket 'sagemaker-example-files-prod-{region}' exists",
            "Check that the file path 'datasets/tabular/synthetic/churn.txt' is correct",
            "Ensure you're using the correct AWS region",
            "Verify the file hasn't been moved or deleted"
        ]
    )

except PermissionError as e:
    FALLBACK_STATUS['churn_s3'] = True
    churn_df = get_fallback_churn_data()
    print_fallback_warning(
        data_source=f"S3: {s3_data_path}",
        error=e,
        resolution_steps=[
            "Check your IAM role has s3:GetObject permission for this bucket",
            "Verify there's no bucket policy blocking access",
            "Ensure your SageMaker execution role includes S3 read permissions",
            "Contact your AWS administrator to grant necessary permissions"
        ]
    )

except Exception as e:
    FALLBACK_STATUS['churn_s3'] = True
    churn_df = get_fallback_churn_data()
    
    error_str = str(e).lower()
    
    if 'nosuchbucket' in error_str or 'nosuchkey' in error_str or 'not found' in error_str:
        resolution_steps = [
            "Verify the S3 bucket exists in your region",
            "Check that the file path is correct",
            "Ensure you're using the correct AWS region",
            f"Current region: {region}"
        ]
    elif 'accessdenied' in error_str or 'forbidden' in error_str or 'unauthorized' in error_str:
        resolution_steps = [
            "Check your IAM role has s3:GetObject permission",
            "Verify there's no bucket policy blocking access",
            "Ensure VPC endpoints allow S3 access if using private subnets",
            "Review your SageMaker execution role permissions"
        ]
    else:
        resolution_steps = [
            "Check your network connectivity",
            "Verify AWS credentials are configured correctly",
            "Ensure the S3 bucket and file exist",
            "Review IAM permissions for S3 access",
            f"Error type: {type(e).__name__}"
        ]
    
    print_fallback_warning(
        data_source=f"S3: s3://sagemaker-example-files-prod-{region}/datasets/tabular/synthetic/churn.txt",
        error=e,
        resolution_steps=resolution_steps
    )

# Display the data regardless of source
churn_df.head(3)

In [0]:
-- Query the churn table from Glue Data Catalog
SELECT *
FROM churn_df
ORDER BY account_length DESC
LIMIT 3

## 3. Work with Spark - Processing Data at Scale

For large-scale data processing, SageMaker Notebooks provide integrated Spark capabilities through Spark Connect. You can write both PySpark and SparkSQL to process data at massive distributed scale on a rapid autoscaling ephemeral infrastructure. 

The notebook environment comes with a pre-configured Spark session accessible via the `spark` variable, no import is needed.

Spark status, SparkUI and Spark Logs can be found in the far bottom-right of the screen in the status menu (typically showing "Ready ^"). 


### 3.1 Running PySpark (with Fallback)

Let's read the churn data using PySpark and perform some transformations using the PySpark DataFrame API.


In [0]:
# Read data using Spark with fallback handling
churn_spark_df = None
FALLBACK_STATUS['spark_session'] = False

try:
    # Check if spark session is available
    if 'spark' not in dir() or spark is None:
        raise RuntimeError("Spark session is not available")
    
    # Create Spark DataFrame from pandas
    churn_spark_df = spark.createDataFrame(churn_df)
    
    # Display schema and preview the data
    print("✅ Successfully created Spark DataFrame")
    churn_spark_df.printSchema()
    
except Exception as e:
    FALLBACK_STATUS['spark_session'] = True
    
    print("\n" + "="*80)
    print("⚠️  SPARK FALLBACK MODE ACTIVATED")
    print("="*80)
    print(f"\n❌ Failed to initialize Spark: {type(e).__name__}: {str(e)[:200]}")
    print("\n✅ Continuing with pandas DataFrame for local processing.")
    print("\n🔧 To resolve this issue:")
    print("   1. Ensure you're running in a SageMaker environment with Spark support")
    print("   2. Check that Spark Connect is properly configured")
    print("   3. Verify your notebook instance has Spark capabilities enabled")
    print("   4. Check the Spark status in the bottom-right status menu")
    print("\n📝 After addressing the issue, re-run this cell.")
    print("="*80 + "\n")
    
    # Create a mock object that allows the notebook to continue
    print("\n📊 Displaying pandas DataFrame schema instead:")
    print(f"Columns: {list(churn_df.columns)}")
    print(f"Shape: {churn_df.shape}")
    print(churn_df.dtypes)

# Display preview
if churn_spark_df is not None:
    churn_spark_df.limit(3)

In [0]:
# Create enriched DataFrame with fallback handling
enriched_df = None
enriched_pandas_fallback = None

try:
    if churn_spark_df is not None and not FALLBACK_STATUS['spark_session']:
        from pyspark.sql.connect import functions as F
        
        # Create features using a single select statement
        enriched_df = churn_spark_df.select(
            "*",
            (F.col("day_mins") + F.col("eve_mins") + F.col("night_mins") + F.col("intl_mins")).alias("total_minutes"),
            (F.col("day_charge") + F.col("eve_charge") + F.col("night_charge") + F.col("intl_charge")).alias("total_charges")
        )
        
        # Show summary statistics for new features by churn status
        result = enriched_df.groupBy("churn").agg(
            F.count("*").alias("count"),
            F.round(F.avg("total_minutes"), 2).alias("avg_total_minutes"),
            F.round(F.avg("total_charges"), 2).alias("avg_total_charges"),
            F.round(F.avg("custserv_calls"), 2).alias("avg_custserv_calls")
        )
        print("✅ Successfully created enriched Spark DataFrame")
        result
    else:
        raise RuntimeError("Spark session not available, using pandas fallback")
        
except Exception as e:
    print("\n⚠️ Using pandas fallback for enriched data processing")
    print(f"   Reason: {str(e)[:100]}\n")
    
    # Create enriched data using pandas
    enriched_pandas_fallback = churn_df.copy()
    enriched_pandas_fallback['total_minutes'] = (
        enriched_pandas_fallback['day_mins'] + 
        enriched_pandas_fallback['eve_mins'] + 
        enriched_pandas_fallback['night_mins'] + 
        enriched_pandas_fallback['intl_mins']
    )
    enriched_pandas_fallback['total_charges'] = (
        enriched_pandas_fallback['day_charge'] + 
        enriched_pandas_fallback['eve_charge'] + 
        enriched_pandas_fallback['night_charge'] + 
        enriched_pandas_fallback['intl_charge']
    )
    
    # Show summary statistics using pandas
    result = enriched_pandas_fallback.groupby('churn').agg({
        'state': 'count',
        'total_minutes': 'mean',
        'total_charges': 'mean',
        'custserv_calls': 'mean'
    }).round(2)
    result.columns = ['count', 'avg_total_minutes', 'avg_total_charges', 'avg_custserv_calls']
    
    # Set enriched_df to the pandas version for downstream compatibility
    enriched_df = enriched_pandas_fallback
    
    print("📊 Summary statistics (using pandas):")
    display(result)

### 3.2 Running Spark SQL from SQL Cells (with Fallback)

You can also use an SQL cell with Spark to query and transform data. This is particularly useful when you prefer SQL syntax for complex analytics. The results of these queries are stored in the Spark dataframe variable specified at the bottom left of the SQL cell.


In [0]:
# Prepare data for SQL query - works with both Spark and pandas
try:
    if not FALLBACK_STATUS['spark_session'] and 'spark' in dir():
        # Use Spark SQL
        churn_spark_df.createOrReplaceTempView('churn_df_view')
        
        query_result = spark.sql("""
            SELECT 
                state,
                CAST(day_charge AS DOUBLE) as day_charge,
                CAST(eve_charge AS DOUBLE) as eve_charge,
                CAST(night_charge AS DOUBLE) as night_charge,
                CAST(intl_charge AS DOUBLE) as intl_charge,
                churn
            FROM churn_df_view
            WHERE churn = 'True.'
            ORDER BY state
            LIMIT 3
        """)
        print("✅ Spark SQL query executed successfully")
        query_result
    else:
        raise RuntimeError("Using pandas fallback")
        
except Exception as e:
    print("\n⚠️ Using pandas/DuckDB fallback for SQL query")
    print(f"   Reason: {str(e)[:100]}\n")
    
    # Use DuckDB for SQL on pandas DataFrame
    import duckdb
    
    query_result = duckdb.sql("""
        SELECT 
            state,
            CAST(day_charge AS DOUBLE) as day_charge,
            CAST(eve_charge AS DOUBLE) as eve_charge,
            CAST(night_charge AS DOUBLE) as night_charge,
            CAST(intl_charge AS DOUBLE) as intl_charge,
            churn
        FROM churn_df
        WHERE churn = 'True.'
        ORDER BY state
        LIMIT 3
    """).df()
    
    print("📊 Query result (using DuckDB on pandas):")
    display(query_result)

### 3.3 Querying PySpark Dataframes with SQL cells (with Fallback)

When running against Athena Spark the SQL cell uses SparkSQL under the hood. Therefore any dataframe you create needs to be registered as a view before SparkSQL can query it. This behavior is the same as running `spark.sql('{query}')` from within PySpark. 
You can use the Variables tool on the left **`{x}`** to view all Spark Dataframes and use the three-dots menu to create a temp view, preview it or it's schema.  
Let's use the `enriched_df` as an example of creating a temp view.


In [0]:
# Create temp view with fallback handling
try:
    if not FALLBACK_STATUS['spark_session'] and hasattr(enriched_df, 'createOrReplaceTempView'):
        enriched_df.createOrReplaceTempView('enriched_df_view')
        print("✅ Successfully created Spark temp view 'enriched_df_view'")
    else:
        raise RuntimeError("enriched_df is not a Spark DataFrame")
        
except Exception as e:
    print("\n⚠️  Spark temp view not created - using pandas DataFrame directly")
    print(f"   Reason: {str(e)[:100]}")
    print("\n   For SQL queries, use DuckDB syntax with the pandas DataFrame 'enriched_df'")
    print("   Example: SELECT * FROM enriched_df LIMIT 3")

In [0]:
# Query enriched data with fallback
try:
    if not FALLBACK_STATUS['spark_session'] and 'spark' in dir():
        result = spark.sql("SELECT * FROM enriched_df_view LIMIT 3")
        print("✅ Spark SQL query on temp view executed successfully")
        result
    else:
        raise RuntimeError("Using pandas fallback")
        
except Exception as e:
    print("\n⚠️ Using DuckDB fallback for querying enriched data")
    import duckdb
    
    # Ensure enriched_df is a pandas DataFrame
    if hasattr(enriched_df, 'toPandas'):
        enriched_df_pandas = enriched_df.limit(1000).toPandas()
    else:
        enriched_df_pandas = enriched_df if isinstance(enriched_df, pd.DataFrame) else enriched_pandas_fallback
    
    result = duckdb.sql("SELECT * FROM enriched_df_pandas LIMIT 3").df()
    print("📊 Query result (using DuckDB):")
    display(result)

### 3.4 Working with Iceberg and S3 Tables (with Fallback)

SageMaker notebooks can work with data in both Glue Catalog and in S3 Tables. You can write to iceberg in the Data Catalog to Standard S3 as well as S3 Tables. 

#### Creating Iceberg Tables in Glue Data Catalog backed by General Purpose Buckets

Use the following code to create an Iceberg table in Glue Data Catalog on a standard S3 bucket


In [0]:
# Drop existing table with fallback handling
try:
    if not FALLBACK_STATUS['spark_session'] and 'spark' in dir():
        spark.sql("DROP TABLE IF EXISTS default.enriched_table")
        print("✅ Successfully dropped existing table (if it existed)")
    else:
        raise RuntimeError("Spark not available")
        
except Exception as e:
    print("\n⚠️  Could not drop table via Spark SQL")
    print(f"   Reason: {str(e)[:100]}")
    print("   This is OK if you're using fallback mode - continuing...")

In [0]:
# Create Iceberg table with comprehensive fallback handling
iceberg_table_created = False
FALLBACK_STATUS['enriched_table'] = False

try:
    if FALLBACK_STATUS['spark_session']:
        raise RuntimeError("Spark session not available")
    
    # Import required modules
    from sagemaker_studio import Project
    import uuid
    
    # Initialize project to get S3 storage path
    proj = Project()
    
    # Ensure enriched_df is a Spark DataFrame
    if not hasattr(enriched_df, 'schema'):
        raise RuntimeError("enriched_df is not a Spark DataFrame")
    
    # Extract schema from enriched_df
    schema_ddl = ", ".join([f"{field.name} {field.dataType.simpleString()}" for field in enriched_df.schema.fields])
    
    # Define table details
    database_name = "default"
    table_name = "enriched_table"
    full_table_name = f"`{database_name}`.`{table_name}`"
    
    # Create storage path for the Iceberg table
    storage_path = f"{proj.s3.root}/iceberg_tables/{database_name}/{table_name}/{uuid.uuid4()}"
    
    # Create the Iceberg table with explicit schema
    create_table_sql = f"""
    CREATE TABLE IF NOT EXISTS {full_table_name} 
    ({schema_ddl})
    USING iceberg
    LOCATION '{storage_path}'
    """
    
    print(f"Creating Iceberg table: {full_table_name}")
    print(f"Storage location: {storage_path}")
    spark.sql(create_table_sql)
    
    # Write data to the Iceberg table using writeTo
    enriched_df.writeTo(full_table_name).append()
    
    row_count = enriched_df.count()
    print(f"\n✅ Successfully created Iceberg table '{full_table_name}' and loaded {row_count} rows")
    
    # Verify the table was created
    verification_df = spark.sql(f"SELECT * FROM {full_table_name} LIMIT 5")
    iceberg_table_created = True
    verification_df

except ImportError as e:
    FALLBACK_STATUS['enriched_table'] = True
    print_fallback_warning(
        data_source="Iceberg Table Creation",
        error=e,
        resolution_steps=[
            "Ensure sagemaker_studio package is installed",
            "Run: pip install sagemaker-studio",
            "Verify you're running in a SageMaker environment"
        ]
    )

except Exception as e:
    FALLBACK_STATUS['enriched_table'] = True
    error_str = str(e).lower()
    
    if 'database' in error_str and ('not found' in error_str or 'does not exist' in error_str):
        resolution_steps = [
            "Verify the 'default' database exists in AWS Glue Data Catalog",
            "Create the database: CREATE DATABASE IF NOT EXISTS default",
            "Check your Glue Data Catalog permissions",
            "Ensure Lake Formation permissions are configured correctly"
        ]
    elif 'permission' in error_str or 'access' in error_str or 'denied' in error_str:
        resolution_steps = [
            "Check IAM permissions for Glue Data Catalog access",
            "Verify Lake Formation permissions for table creation",
            "Ensure S3 write permissions for the storage location",
            "Contact your AWS administrator for permission grants"
        ]
    elif 'spark' in error_str or 'session' in error_str:
        resolution_steps = [
            "Ensure Spark session is properly initialized",
            "Check Spark Connect configuration",
            "Verify notebook has Spark capabilities enabled",
            "Check Spark status in the bottom-right status menu"
        ]
    else:
        resolution_steps = [
            "Verify AWS credentials and permissions",
            "Check that required databases exist",
            "Ensure S3 bucket is accessible",
            "Review Spark and Glue configurations",
            f"Error type: {type(e).__name__}"
        ]
    
    print_fallback_warning(
        data_source="Iceberg Table: default.enriched_table",
        error=e,
        resolution_steps=resolution_steps
    )
    
    # Show fallback data
    print("📊 Showing fallback enriched data preview:")
    fallback_enriched = get_fallback_enriched_data()
    display(fallback_enriched.head(5))

### 3.5 Read Iceberg tables from General Purpose S3 Buckets and Glue Data Catalog (with Fallback)


In [0]:
# Read from Iceberg table with fallback handling
read_iceberg = None

try:
    if FALLBACK_STATUS['spark_session'] or FALLBACK_STATUS['enriched_table']:
        raise RuntimeError("Using fallback due to previous errors")
    
    read_iceberg = spark.read.table('`default`.`enriched_table`')
    print("✅ Successfully read from Iceberg table 'default.enriched_table'")
    read_iceberg

except Exception as e:
    error_str = str(e).lower()
    
    if 'table' in error_str and ('not found' in error_str or 'does not exist' in error_str):
        resolution_steps = [
            "Verify the table 'enriched_table' exists in 'default' database",
            "Run the previous cell to create the table first",
            "Check Glue Data Catalog for the table",
            "Ensure you have SELECT permissions on the table"
        ]
    elif 'database' in error_str and ('not found' in error_str or 'does not exist' in error_str):
        resolution_steps = [
            "Verify the 'default' database exists",
            "Create the database: CREATE DATABASE IF NOT EXISTS default",
            "Check Glue Data Catalog permissions"
        ]
    elif 'permission' in error_str or 'access' in error_str:
        resolution_steps = [
            "Check IAM permissions for Glue Data Catalog",
            "Verify Lake Formation SELECT permissions",
            "Ensure S3 read permissions for underlying data"
        ]
    else:
        resolution_steps = [
            "Ensure the table was created successfully",
            "Verify Spark session is active",
            "Check database and table exist",
            "Review permissions for data access"
        ]
    
    print_fallback_warning(
        data_source="Iceberg Table: default.enriched_table",
        error=e,
        resolution_steps=resolution_steps
    )
    
    # Use fallback data
    print("📊 Showing fallback enriched data:")
    read_iceberg = get_fallback_enriched_data()
    display(read_iceberg.head(5))

In [0]:
# Read Iceberg table with Athena SQL - with fallback
try:
    if FALLBACK_STATUS['spark_session'] or FALLBACK_STATUS['enriched_table']:
        raise RuntimeError("Using fallback due to previous errors")
    
    result = spark.sql("SELECT * FROM default.enriched_table LIMIT 3")
    print("✅ Successfully queried Iceberg table with Spark SQL")
    result
    
except Exception as e:
    print("\n⚠️ Using DuckDB fallback for Iceberg table query")
    print(f"   Reason: {str(e)[:100]}\n")
    
    import duckdb
    fallback_data = get_fallback_enriched_data()
    result = duckdb.sql("SELECT * FROM fallback_data LIMIT 3").df()
    print("📊 Query result (using fallback data):")
    display(result)

## 4. Working with S3 Table buckets (with Fallback)

Amazon S3 Tables are optimized for analytics workloads and provide native support for Apache Iceberg table format. They offer better performance, reliability, and cost efficiency for managing tabular data at scale.


### 4.1 Create an S3 Tables Bucket (with Fallback)

To store data in S3 table buckets, three steps are required:
1. Create an S3 table bucket (usually done by your administrator)
2. Create an S3 table Bucket Namespace (same thing as a Glue Database)
3. Create the S3 Table


In [0]:
import boto3

# Initialize S3 Tables variables
s3_tables_bucket = None
bucket_arn = None
FALLBACK_STATUS['s3_tables'] = False

try:
    # Get account ID and region
    account_id = boto3.client('sts').get_caller_identity()['Account']
    region = boto3.Session().region_name
    
    # Desired bucket name
    desired_bucket_name = "sagemakersamples"
    
    s3tables_client = boto3.client('s3tables', region_name=region)
    
    print(f"🔍 Checking for S3 Tables bucket '{desired_bucket_name}'...\n")
    
    # List existing buckets to see if our desired bucket exists
    response = s3tables_client.list_table_buckets()
    existing_buckets = response.get('tableBuckets', [])
    existing_bucket_names = [b['name'] for b in existing_buckets]
    
    # Check if desired bucket already exists
    if desired_bucket_name in existing_bucket_names:
        print(f"✅ Bucket '{desired_bucket_name}' already exists!")
        for bucket in existing_buckets:
            if bucket['name'] == desired_bucket_name:
                bucket_arn = bucket['arn']
                print(f"  ARN: {bucket_arn}")
                break
        s3_tables_bucket = desired_bucket_name
        
    else:
        # Bucket doesn't exist, try to create it
        print(f"Bucket '{desired_bucket_name}' not found. Attempting to create it...")
        
        try:
            create_response = s3tables_client.create_table_bucket(
                name=desired_bucket_name
            )
            bucket_arn = create_response['arn']
            s3_tables_bucket = desired_bucket_name
            print(f"✅ Successfully created new S3 Tables bucket: {desired_bucket_name}")
            print(f"  ARN: {bucket_arn}")
            
        except s3tables_client.exceptions.ConflictException:
            print(f"✅ Bucket '{desired_bucket_name}' already exists (created by another process)")
            s3_tables_bucket = desired_bucket_name
            
        except Exception as create_error:
            error_msg = str(create_error)
            
            if "more buckets than are allowed" in error_msg:
                print(f"⚠️ Cannot create new bucket - account quota limit reached")
                if existing_buckets:
                    s3_tables_bucket = existing_buckets[0]['name']
                    bucket_arn = existing_buckets[0]['arn']
                    print(f"  Using existing bucket: {s3_tables_bucket}")
                else:
                    raise
            else:
                raise
    
    # Print summary
    if s3_tables_bucket:
        print(f"\n" + "="*60)
        print(f"S3 Tables Bucket: {s3_tables_bucket}")
        print(f"Catalog ID: {account_id}:s3tablescatalog/{s3_tables_bucket}")
        print("="*60)

except Exception as e:
    FALLBACK_STATUS['s3_tables'] = True
    error_str = str(e).lower()
    
    if 'accessdenied' in error_str or 'not authorized' in error_str:
        resolution_steps = [
            "Check IAM permissions for s3tables:ListTableBuckets",
            "Verify s3tables:CreateTableBucket permission if creating new bucket",
            "Ensure your role has S3 Tables access",
            "Contact your AWS administrator for permission grants"
        ]
    elif 'region' in error_str or 'endpoint' in error_str:
        resolution_steps = [
            "S3 Tables may not be available in your region",
            f"Current region: {region}",
            "Try using a supported region (e.g., us-east-1, us-west-2)",
            "Check AWS documentation for S3 Tables availability"
        ]
    else:
        resolution_steps = [
            "Verify AWS credentials are configured correctly",
            "Check network connectivity to AWS services",
            "Ensure S3 Tables service is available in your region",
            "Review IAM permissions for S3 Tables operations"
        ]
    
    print_fallback_warning(
        data_source="S3 Tables Bucket",
        error=e,
        resolution_steps=resolution_steps
    )
    
    print("📊 S3 Tables operations will be skipped. Using local data instead.")

In [0]:
# Create namespace in S3 Tables bucket with fallback
namespace_name = "getting_started"
namespace_created = False

try:
    if FALLBACK_STATUS['s3_tables'] or s3_tables_bucket is None:
        raise RuntimeError("S3 Tables bucket not available")
    
    import boto3
    
    s3tables_client = boto3.client('s3tables')
    
    # Get the bucket ARN
    if bucket_arn is None:
        region = boto3.Session().region_name
        account_id = boto3.client('sts').get_caller_identity()['Account']
        bucket_arn = f"arn:aws:s3tables:{region}:{account_id}:bucket/{s3_tables_bucket}"
    
    print(f"Creating namespace '{namespace_name}' in S3 Tables bucket '{s3_tables_bucket}'...")
    
    try:
        response = s3tables_client.create_namespace(
            tableBucketARN=bucket_arn,
            namespace=[namespace_name]
        )
        print(f"✅ Successfully created namespace: {namespace_name}")
        namespace_created = True
        
    except s3tables_client.exceptions.ConflictException:
        print(f"✅ Namespace '{namespace_name}' already exists - proceeding")
        namespace_created = True
        
    except Exception as e:
        if "ConflictException" in str(e) or "already exists" in str(e).lower():
            print(f"✅ Namespace '{namespace_name}' already exists - proceeding")
            namespace_created = True
        else:
            raise
    
    print(f"\n✅ Namespace ready!")
    print(f"Full path: {s3_tables_bucket}.{namespace_name}")
    print(f"Use in Spark: `{s3_tables_bucket}`.`{namespace_name}`.`table_name`")

except Exception as e:
    FALLBACK_STATUS['s3_tables'] = True
    error_str = str(e).lower()
    
    if 'permission' in error_str or 'access' in error_str or 'denied' in error_str:
        resolution_steps = [
            "Check IAM permissions for s3tables:CreateNamespace",
            "Verify your role has S3 Tables namespace management access",
            "Ensure Lake Formation permissions are configured",
            "Contact your AWS administrator for permission grants"
        ]
    elif 's3 tables' in error_str or 'not available' in error_str:
        resolution_steps = [
            "S3 Tables bucket was not created successfully",
            "Re-run the previous cell to create the bucket",
            "Check if S3 Tables is available in your region"
        ]
    else:
        resolution_steps = [
            "Verify the S3 Tables bucket exists",
            "Check IAM permissions for namespace creation",
            "Ensure network connectivity to AWS services",
            f"Error: {type(e).__name__}"
        ]
    
    print_fallback_warning(
        data_source=f"S3 Tables Namespace: {namespace_name}",
        error=e,
        resolution_steps=resolution_steps
    )
    
    print("📊 S3 Tables namespace operations will be skipped.")

In [0]:
# Write enriched DataFrame to S3 Tables with fallback
s3_table_created = False

try:
    if FALLBACK_STATUS['s3_tables'] or not namespace_created:
        raise RuntimeError("S3 Tables namespace not available")
    
    import boto3
    
    table_name = "enriched_table"
    
    # Get the data to write
    if hasattr(enriched_df, 'toPandas'):
        print(f"Converting Spark DataFrame to Pandas...")
        pandas_df = enriched_df.limit(5000).toPandas()
    elif isinstance(enriched_df, pd.DataFrame):
        pandas_df = enriched_df.head(5000)
    else:
        pandas_df = get_fallback_enriched_data()
    
    print(f"Shape: {pandas_df.shape}")
    print(f"Columns: {list(pandas_df.columns)}")
    
    # Write to S3 location for S3 Tables
    from sagemaker_studio import Project
    proj = Project()
    
    s3_location = f"{proj.s3.root}/s3tables_data/{namespace_name}/{table_name}/"
    print(f"\nWriting data to: {s3_location}")
    
    # Write as Parquet to S3
    pandas_df.to_parquet(s3_location + "data.parquet", index=False)
    
    print(f"✅ Data written successfully!")
    s3_table_created = True
    
    print(f"\nOnce the table is created by admin, you can query with:")
    print(f"  spark.read.table('`{s3_tables_bucket}`.`{namespace_name}`.`{table_name}`')")

except ImportError as e:
    FALLBACK_STATUS['s3_tables'] = True
    print_fallback_warning(
        data_source="S3 Tables Data Write",
        error=e,
        resolution_steps=[
            "Ensure sagemaker_studio package is installed",
            "Run: pip install sagemaker-studio",
            "Verify you're running in a SageMaker environment"
        ]
    )

except Exception as e:
    FALLBACK_STATUS['s3_tables'] = True
    error_str = str(e).lower()
    
    if 'permission' in error_str or 'access' in error_str or 'denied' in error_str:
        resolution_steps = [
            "Check IAM permissions for S3 write access",
            "Verify s3:PutObject permission for the target bucket",
            "Ensure your SageMaker execution role has S3 write access",
            "Contact your AWS administrator for permission grants"
        ]
    elif 'namespace' in error_str or 'not available' in error_str:
        resolution_steps = [
            "S3 Tables namespace was not created successfully",
            "Re-run the previous cells to create bucket and namespace",
            "Check S3 Tables availability in your region"
        ]
    else:
        resolution_steps = [
            "Verify S3 Tables bucket and namespace exist",
            "Check S3 write permissions",
            "Ensure network connectivity",
            f"Error: {type(e).__name__}"
        ]
    
    print_fallback_warning(
        data_source="S3 Tables Data Write",
        error=e,
        resolution_steps=resolution_steps
    )
    
    print("📊 Showing fallback enriched data instead:")
    display(get_fallback_enriched_data().head(5))

### 4.2 Querying S3 Tables with Athena SQL (with Fallback)

To query S3 Tables with SQL you need to use the three-part namespace with quotes around the catalog name that includes the "s3tablescatalog/" prefix.


In [0]:
# Query S3 Tables with fallback handling
try:
    if FALLBACK_STATUS['s3_tables'] or not s3_table_created:
        raise RuntimeError("S3 Tables not available - using fallback")
    
    if FALLBACK_STATUS['spark_session']:
        raise RuntimeError("Spark session not available")
    
    # Query using Spark SQL with S3 Tables catalog
    s3_tables_query = f'SELECT * FROM `{s3_tables_bucket}`.`{namespace_name}`.`enriched_table` LIMIT 5'
    print(f"Executing query: {s3_tables_query}")
    
    result = spark.sql(s3_tables_query)
    print("✅ Successfully queried S3 Tables")
    result

except Exception as e:
    error_str = str(e).lower()
    
    if 'table' in error_str and ('not found' in error_str or 'does not exist' in error_str):
        resolution_steps = [
            "Verify the S3 Table was created successfully",
            "Check that the table exists in the S3 Tables namespace",
            "The table may need to be created by an administrator",
            "Run the previous cell to write data first"
        ]
    elif 'permission' in error_str or 'access' in error_str:
        resolution_steps = [
            "Check IAM permissions for S3 Tables read access",
            "Verify Lake Formation permissions for S3 Tables",
            "Ensure your role can access the S3 Tables catalog"
        ]
    elif 's3 tables' in error_str or 'not available' in error_str or 'fallback' in error_str:
        resolution_steps = [
            "S3 Tables was not set up successfully in previous cells",
            "Re-run the S3 Tables bucket and namespace creation cells",
            "Check S3 Tables availability in your region"
        ]
    else:
        resolution_steps = [
            "Verify S3 Tables setup completed successfully",
            "Check Spark session is active",
            "Review S3 Tables permissions",
            f"Error: {type(e).__name__}"
        ]
    
    print_fallback_warning(
        data_source="S3 Tables Query",
        error=e,
        resolution_steps=resolution_steps
    )
    
    # Use fallback data with DuckDB
    print("📊 Querying fallback data with DuckDB instead:")
    import duckdb
    fallback_data = get_fallback_enriched_data()
    result = duckdb.sql("SELECT * FROM fallback_data LIMIT 5").df()
    display(result)

### 4.3 Querying S3 Tables with Spark SQL (with Fallback)

To query S3 Tables with Spark SQL, use Spark's 3-part namespace with backticks for identifiers that may contain special characters.


In [0]:
# Query S3 Tables with Spark SQL - with fallback
try:
    if FALLBACK_STATUS['s3_tables'] or FALLBACK_STATUS['spark_session']:
        raise RuntimeError("S3 Tables or Spark not available")
    
    query = f"SELECT * FROM `{s3_tables_bucket}`.`{namespace_name}`.`enriched_table` LIMIT 3"
    print(f"Executing: {query}")
    
    result = spark.sql(query)
    print("✅ Spark SQL query on S3 Tables executed successfully")
    result

except Exception as e:
    print("\n⚠️ Using DuckDB fallback for S3 Tables Spark SQL query")
    print(f"   Reason: {str(e)[:100]}\n")
    
    import duckdb
    fallback_data = get_fallback_enriched_data()
    result = duckdb.sql("SELECT * FROM fallback_data LIMIT 3").df()
    print("📊 Query result (using fallback data):")
    display(result)

### 4.4 Joining Glue Catalog Tables with S3 Tables (with Fallback)

To join Glue Catalog tables with S3 tables, the best practice is to specify the full name for each table using the appropriate catalog prefix.


In [0]:
# Join Glue Catalog and S3 Tables - with fallback
try:
    if FALLBACK_STATUS['s3_tables'] or FALLBACK_STATUS['spark_session'] or FALLBACK_STATUS['enriched_table']:
        raise RuntimeError("Required data sources not available")
    
    union_query = f"""
    SELECT * FROM `spark_catalog`.`default`.`enriched_table`
    UNION ALL
    SELECT * FROM `{s3_tables_bucket}`.`{namespace_name}`.`enriched_table`
    LIMIT 6
    """
    print(f"Executing union query...")
    
    result = spark.sql(union_query)
    print("✅ Successfully joined Glue Catalog and S3 Tables")
    result

except Exception as e:
    print("\n⚠️  Using pandas fallback for cross-catalog join")
    print(f"   Reason: {str(e)[:100]}\n")
    
    fallback_data = get_fallback_enriched_data()
    
    # Simulate a union by concatenating dataframes with pandas
    glue_sample = fallback_data.head(3).copy()
    glue_sample['_source'] = 'glue_catalog'
    
    s3tables_sample = fallback_data.head(3).copy()
    s3tables_sample['_source'] = 's3_tables'
    
    result = pd.concat([glue_sample, s3tables_sample], ignore_index=True)
    
    print("📊 Simulated union query result (using fallback data):")
    print("   Note: In production, this would join data from Glue Catalog and S3 Tables")
    print("   Added '_source' column to show which catalog each row would come from")
    display(result)

## 5. Using Pandas on Spark (with Fallback)

For ML pre-processing you can invoke pandas directly on Spark to distribute and increase the performance of pandas workloads.


In [0]:
# Enable pandas API on Spark with fallback
enriched_psdf = None

try:
    if FALLBACK_STATUS['spark_session']:
        raise RuntimeError("Spark session not available")
    
    if not hasattr(enriched_df, 'pandas_api'):
        raise RuntimeError("enriched_df is not a Spark DataFrame")
    
    import pyspark.pandas as ps
    
    enriched_psdf = enriched_df.pandas_api()
    print("✅ Successfully created pandas-on-Spark DataFrame")
    enriched_psdf.head()

except Exception as e:
    print("\n⚠️ Using standard pandas fallback")
    print(f"   Reason: {str(e)[:100]}\n")
    
    # Use regular pandas
    if isinstance(enriched_df, pd.DataFrame):
        enriched_psdf = enriched_df
    else:
        enriched_psdf = get_fallback_enriched_data()
    
    print("📊 Using standard pandas DataFrame:")
    print("   Note: For large datasets, pandas-on-Spark provides distributed processing")
    display(enriched_psdf.head())

### 5.1 Returning Data to Local Environment (with Fallback)

While Spark is great for processing large datasets, you often want to bring small result sets back to the local notebook for training models on local GPUs.


In [0]:
# Convert to pandas with fallback
enriched_pandas = None

try:
    if FALLBACK_STATUS['spark_session']:
        raise RuntimeError("Spark session not available")
    
    if hasattr(enriched_df, 'limit') and hasattr(enriched_df, 'toPandas'):
        # Spark DataFrame
        enriched_pandas = enriched_df.limit(50).toPandas()
        print(f"✅ Converted {len(enriched_pandas)} rows from Spark to pandas DataFrame")
    else:
        raise RuntimeError("enriched_df is not a Spark DataFrame")

except Exception as e:
    print("\n⚠️ Using local pandas data")
    print(f"   Reason: {str(e)[:100]}\n")
    
    if isinstance(enriched_df, pd.DataFrame):
        enriched_pandas = enriched_df.head(50)
    else:
        enriched_pandas = get_fallback_enriched_data().head(50)
    
    print(f"📊 Using {len(enriched_pandas)} rows of local pandas data")

display(enriched_pandas)

In [0]:
# Convert pandas to Spark with fallback
enriched_sdf = None

try:
    if FALLBACK_STATUS['spark_session']:
        raise RuntimeError("Spark session not available")
    
    enriched_sdf = spark.createDataFrame(enriched_pandas)
    print("✅ Successfully converted pandas DataFrame to Spark DataFrame")
    enriched_sdf

except Exception as e:
    print("\n⚠️  Cannot convert to Spark DataFrame - Spark not available")
    print(f"   Reason: {str(e)[:100]}\n")
    print("📊 Keeping data as pandas DataFrame for local processing:")
    display(enriched_pandas.head())

### 5.2 Configuring Spark (with Fallback)

You can configure Spark settings using the `spark.conf` API. There are no magic commands - use the `spark` object directly.


In [0]:
# Configure Spark with fallback
try:
    if FALLBACK_STATUS['spark_session']:
        raise RuntimeError("Spark session not available")
    
    # Set Spark configuration
    spark.conf.set("spark.sql.shuffle.partitions", "10")
    spark.conf.set("spark.sql.adaptive.enabled", "true")
    
    # Get configuration values
    shuffle_partitions = spark.conf.get("spark.sql.shuffle.partitions")
    adaptive_enabled = spark.conf.get("spark.sql.adaptive.enabled")
    
    print("✅ Spark configuration updated successfully")
    print(f"   shuffle.partitions: {shuffle_partitions}")
    print(f"   adaptive.enabled: {adaptive_enabled}")

except Exception as e:
    print("\n⚠️  Spark configuration not available")
    print(f"   Reason: {str(e)[:100]}\n")
    print("📋 Spark configuration options (for when Spark is available):")
    print("   - spark.sql.shuffle.partitions: Controls number of partitions for shuffles")
    print("   - spark.sql.adaptive.enabled: Enables adaptive query execution")
    print("   - spark.executor.memory: Memory per executor")
    print("   - spark.driver.memory: Memory for driver")

## 6. Working with Connections

You can connect to RDS, Redshift, Snowflake and many others and query them with Athena (SQL) and Athena (Spark). To connect use the Connections tool on the main menu.


## Fallback Status Summary

This cell provides a summary of which data sources used fallback data during execution.


In [0]:
# Print fallback status summary
print("="*60)
print("📊 FALLBACK STATUS SUMMARY")
print("="*60)

all_ok = True
for source, used_fallback in FALLBACK_STATUS.items():
    status = "⚠️  FALLBACK" if used_fallback else "✅ OK"
    print(f"   {source}: {status}")
    if used_fallback:
        all_ok = False

print("="*60)

if all_ok:
    print("\n🎉 All data sources loaded successfully!")
else:
    print("\n⚠️  Some data sources used fallback data.")
    print("   Review the warnings above for instructions on resolving issues.")
    print("   After addressing issues, re-run the affected cells.")

print("\n" + "="*60)

## Summary and Next Steps

Congratulations! You've now explored the key features of Notebooks in Amazon SageMaker Unified Studio:

**What You've Learned:**

1. **Polyglot Programming** - Seamlessly work with both Python and DuckDB, choosing the best tool for each task

2. **Multi-Engine Support** - Query data from AWS Glue Data Catalog using Athena SQL for fast, serverless analytics

3. **Spark Integration** - Process large-scale data with PySpark and Spark SQL, leveraging distributed computing for big data workloads

4. **Multi-Source Data** - Combine data from various sources (Glue Catalog, S3, and potentially Snowflake) into unified analyses

5. **Flexible Workflows** - Move data between engines (Spark to pandas), write results to S3, and create rich visualizations

6. **Graceful Fallbacks** - The notebook handles errors gracefully and provides clear instructions for resolution

**Next Steps:**

- Review the Fallback Status Summary above to see if any data sources need attention
- Try working with your own datasets in the Glue Data Catalog
- Experiment with more complex Spark transformations and optimizations
- Build end-to-end data pipelines combining multiple sources
- Explore machine learning workflows using the prepared datasets
- Configure additional connections for external data sources
- Schedule a notebook with the SageMaker Unified Studio Notebook operator in Workflows

SageMaker Notebooks provide a powerful, unified environment for all your data work - from exploration to production pipelines.